# 00 — Pull the raw data

**Takes in:** nothing (downloads from the web)

**Does:** downloads the two Opportunity Insights Atlas tables this project uses, into `data/`. Files already present are left alone, so re-running the notebook is cheap. Prints the shape and a few key columns of each file so the download can be sanity-checked.

**Outputs:** `data/tract_outcomes_simple.csv` (Atlas table 4, ~33 MB), `data/tract_covariates.csv` (Atlas table 9, ~24 MB)

Source: [Opportunity Insights data library](https://opportunityinsights.org/data/). Unit of analysis is the 2010 Census tract, keyed to where a child grew up; outcomes are for the 1978–1983 birth cohorts, measured in adulthood from federal tax records.


In [1]:
import os
import sys
import urllib.request

import pandas as pd

## code/ is on the path so the shared constants in utils.py can be imported
sys.path.insert(0, os.path.abspath("."))
import utils

print("project root:", utils.PROJ_DIR)
print("data dir:    ", utils.DATA_DIR)

project root: /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility
data dir:     /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/data


## Functions

In [2]:
def download_if_missing(filename, url, data_dir):
    """Download `url` to `data_dir/filename` unless the file is already there.

    Returns the absolute path. Downloads to a .part file first so that an
    interrupted download never leaves a truncated CSV that looks complete.
    """
    dest = os.path.join(data_dir, filename)
    if os.path.exists(dest):
        print("{:32s} already present ({:.1f} MB)".format(
            filename, os.path.getsize(dest) / 1e6))
        return dest

    print("{:32s} downloading from {}".format(filename, url))
    tmp = dest + ".part"
    urllib.request.urlretrieve(url, tmp)
    os.rename(tmp, dest)
    print("{:32s} downloaded ({:.1f} MB)".format(
        filename, os.path.getsize(dest) / 1e6))
    return dest


def describe_raw(path, id_cols, key_vars):
    """Print the shape of a raw file plus missingness on the columns we rely on."""
    df = pd.read_csv(path, usecols=id_cols + key_vars)
    print("\n{}: {:,} rows x {} cols read".format(os.path.basename(path), len(df), df.shape[1]))
    print("  duplicate tract ids:", int(df.duplicated(subset=id_cols).sum()))
    for var in key_vars:
        n_missing = int(df[var].isna().sum())
        print("  {:28s} missing {:,} ({:.1%})".format(var, n_missing, n_missing / len(df)))
    return df

## Download

In [3]:
os.makedirs(utils.DATA_DIR, exist_ok=True)

paths = {}
for filename, url in utils.DATA_SOURCES.items():
    paths[filename] = download_if_missing(filename, url, utils.DATA_DIR)

tract_outcomes_simple.csv        already present (34.4 MB)
tract_covariates.csv             already present (24.9 MB)


## Check what came down

In [4]:
outcomes_head = describe_raw(
    paths["tract_outcomes_simple.csv"],
    utils.ID_COLS,
    [utils.MOBILITY_VAR, utils.COUNT_VAR],
)

covariates_head = describe_raw(
    paths["tract_covariates.csv"],
    utils.ID_COLS,
    ["popdensity2000", "poor_share2000", "singleparent_share2000"],
)


tract_outcomes_simple.csv: 73,278 rows x 5 cols read
  duplicate tract ids: 0
  kfr_pooled_pooled_p25        missing 1,264 (1.7%)
  pooled_pooled_count          missing 1,328 (1.8%)

tract_covariates.csv: 74,044 rows x 6 cols read
  duplicate tract ids: 0
  popdensity2000               missing 1,305 (1.8%)
  poor_share2000               missing 1,628 (2.2%)
  singleparent_share2000       missing 1,666 (2.3%)


In [5]:
outcomes_head.head()

,state,county,tract,kfr_pooled_pooled_p25,pooled_pooled_count
0,1,1,20100,0.367813,228.14854
1,1,1,20200,0.316781,391.91345
2,1,1,20300,0.373485,394.79211
3,1,1,20400,0.421511,388.38309
4,1,1,20500,0.433415,419.08334


In [6]:
covariates_head.head()

,state,county,tract,popdensity2000,poor_share2000,singleparent_share2000
0,1,1,20100,195.72380,0.126816,0.250980
1,1,1,20200,566.38141,0.227058,0.392523
2,1,1,20300,624.19684,0.076640,0.244856
3,1,1,20400,713.80396,0.045485,0.190722
4,1,1,20500,529.93030,0.036792,0.168000


The two files are keyed the same way (`state`, `county`, `tract`) with no duplicate tract ids, so they can be joined one-to-one in [`01_merge.ipynb`](01_merge.ipynb).
